In [ ]:
import duckdb
LAB = "../data/mimiciv-3.1/hosp/labevents.csv"

display(duckdb.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{LAB}')").df())
display(duckdb.sql(f"SELECT COUNT(*) FROM read_csv_auto('{LAB}')").df())
display(duckdb.sql(f"SELECT * FROM read_csv_auto('{LAB}') LIMIT 5").df())

In [ ]:
duckdb.sql(f"""
    COPY (SELECT subject_id, hadm_id, itemid, charttime, value, valuenum, valueuom, ref_range_lower, 
        ref_range_upper, flag FROM read_csv_auto('{LAB}'))
    TO '../data/processed/labevents.parquet' (FORMAT PARQUET)
""")

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

HOSP = Path("../data/mimiciv-3.1/hosp")
labitems = pd.read_csv(HOSP / "d_labitems.csv")

display(labitems.shape, labitems.head(), labitems.info(), labitems.isna().sum(),
        labitems.nunique(), labitems.describe(), labitems["fluid"].value_counts(), 
        labitems["category"].value_counts(), labitems["label"].value_counts())

In [ ]:
LABP = "../data/processed/labevents.parquet"
# displays total number of patients and from that total the number of patients tha have hospital stays
display(duckdb.sql(f"SELECT COUNT(*) AS total, COUNT(hadm_id) AS with_stay FROM '{LABP}'").df())

stays = pd.read_parquet("../data/processed/admissions_65plus.parquet")
# filter lab events to contain only for the patients are 65 plus
labs_65 = duckdb.sql(f"SELECT * FROM '{LABP}' WHERE hadm_id IN (SELECT hadm_id FROM stays)").df()
labs_65.to_parquet("../data/processed/labevents_65plus.parquet", index=False)

In [ ]:
n_stays = len(stays)   

# what % of the stays have a labevent of a specific type, sorted from most common to least
coverage = labs_65.groupby("itemid")["hadm_id"].nunique().reset_index(name="n_stays")
coverage["pct_stays"] = coverage["n_stays"] / n_stays * 100
coverage = coverage.sort_values("pct_stays", ascending=False)

coverage = pd.merge(coverage, labitems, on="itemid", how="left")
display(coverage.head(30))

# top 25 most common labevents 
top = coverage.head(25).copy()
top["name"] = top["label"].fillna("unknown") + " (" + top["fluid"] + ")"

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(top["name"], top["pct_stays"], color="steelblue")
ax.invert_yaxis()                    
ax.set_xlim(0, 100)
ax.set_xlabel("% of stays with this test")
ax.set_title("25 most common lab tests, stays of patients aged 65+")
ax.grid(axis="x", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()